In [15]:
import pandas as pd
import glob
import os

print("Starting sensing data analysis...")

# 1️⃣ อ่านทุกไฟล์ CSV ในโฟลเดอร์
folder_path = "data_0/71_sensing_data/cefoxSR1707003701"  # เปลี่ยนเป็น path ของคุณ

# ตรวจสอบว่าโฟลเดอร์มีอยู่หรือไม่
if not os.path.exists(folder_path):
    print(f"Error: Folder '{folder_path}' not found!")
    print("Available sensing data folders:")
    sensing_base = "data_0/71_sensing_data"
    if os.path.exists(sensing_base):
        available_folders = [f for f in os.listdir(sensing_base) if os.path.isdir(os.path.join(sensing_base, f))]
        for folder in available_folders:
            print(f"  {folder}")
        
        if available_folders:
            # ใช้โฟลเดอร์แรกที่มีอยู่
            folder_path = os.path.join(sensing_base, available_folders[0])
            print(f"\nUsing first available folder: {folder_path}")
        else:
            print("No sensing data folders found!")
            folder_path = None
    else:
        print(f"Base sensing data folder '{sensing_base}' not found!")
        folder_path = None

if folder_path and os.path.exists(folder_path):
    all_files = glob.glob(os.path.join(folder_path, "*.csv"))
    print(f"Found {len(all_files)} CSV files in {folder_path}")
    
    if len(all_files) == 0:
        print("No CSV files found in the specified folder!")
    else:
        # อ่านและรวมข้อมูล
        df_list = []
        for file in all_files[:10]:  # เพิ่มจำนวนไฟล์ที่อ่าน
            print(f"Reading: {os.path.basename(file)}")
            try:
                temp_df = pd.read_csv(file)
                # ทำความสะอาดชื่อคอลัมน์ (เอาช่องว่างออก)
                temp_df.columns = temp_df.columns.str.strip()
                
                # ดึงวันที่จากชื่อไฟล์ (เช่น: cefoxSR17070037_activ_20210123075846.csv)
                filename = os.path.basename(file)
                try:
                    # หาตัวเลขวันที่ในชื่อไฟล์ (8 หลักแรกหลังจาก _activ_)
                    if '_activ_' in filename:
                        date_part = filename.split('_activ_')[1][:8]  # เอา 8 หลักแรก
                        file_date = pd.to_datetime(date_part, format='%Y%m%d')
                        temp_df['File_Date'] = file_date.date()
                        print(f"  Extracted date: {file_date.date()}")
                    else:
                        temp_df['File_Date'] = None
                        print(f"  Could not extract date from filename")
                except:
                    temp_df['File_Date'] = None
                    print(f"  Error extracting date from filename")
                
                print(f"  Shape: {temp_df.shape}, Columns: {temp_df.columns.tolist()}")
                df_list.append(temp_df)
            except Exception as e:
                print(f"  Error reading {file}: {e}")
        
        if len(df_list) > 0:
            df = pd.concat(df_list, ignore_index=True)
            print(f"\n=== COMBINED DATA ANALYSIS ===")
            print(f"Combined data shape: {df.shape}")
            print(f"Columns: {df.columns.tolist()}")
            
            # 2️⃣ ทำความสะอาดข้อมูล
            # ตรวจสอบว่ามีคอลัมน์ Measure Date Time หรือไม่
            date_col = None
            for col in df.columns:
                if 'date' in col.lower() and 'time' in col.lower():
                    date_col = col
                    break
            
            if date_col:
                df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
                print(f"Date range from datetime column: {df[date_col].min()} to {df[date_col].max()}")
                df['Date'] = df[date_col].dt.date
            elif 'File_Date' in df.columns and df['File_Date'].notna().any():
                # ใช้วันที่จากชื่อไฟล์
                print("Using dates extracted from filenames")
                df['Date'] = df['File_Date']
                print(f"Date range from filenames: {df['Date'].min()} to {df['Date'].max()}")
            else:
                print("No date information found! Creating sequential dates...")
                # สร้างวันที่ต่อเนื่องตามจำนวนไฟล์
                unique_files = len(df_list)
                base_date = pd.to_datetime('2021-01-23')  # วันที่จากไฟล์แรกที่เห็น
                date_range = pd.date_range(base_date, periods=unique_files, freq='D')
                
                # กำหนดวันที่ให้แต่ละไฟล์
                df['Date'] = None
                start_idx = 0
                for i, (file, temp_df) in enumerate(zip(all_files[:len(df_list)], df_list)):
                    end_idx = start_idx + len(temp_df)
                    df.loc[start_idx:end_idx-1, 'Date'] = date_range[i].date()
                    start_idx = end_idx
                
                print(f"Created sequential dates: {df['Date'].min()} to {df['Date'].max()}")
            
            # ตรวจสอบคอลัมน์ที่มีอยู่จริง
            expected_cols = ['Battery Level', 'Temperature', 'Step', 'Calorie', 'Sleep Hour', 'Sleep Minute']
            available_cols = [col for col in expected_cols if col in df.columns]
            missing_cols = [col for col in expected_cols if col not in df.columns]
            
            print(f"\nAvailable columns: {available_cols}")
            if missing_cols:
                print(f"Missing columns: {missing_cols}")
            
            # แปลงเฉพาะคอลัมน์ที่มีอยู่
            if available_cols:
                for col in available_cols:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
            
            # สร้างคอลัมน์ Total Sleep ถ้ามีข้อมูล Sleep
            if 'Sleep Hour' in df.columns and 'Sleep Minute' in df.columns:
                df['Total Sleep (min)'] = df['Sleep Hour']*60 + df['Sleep Minute']
                print("Created Total Sleep column from Sleep Hour + Sleep Minute")
            elif 'Sleep Hour' in df.columns:
                df['Total Sleep (min)'] = df['Sleep Hour']*60
                print("Created Total Sleep column from Sleep Hour only")
            else:
                df['Total Sleep (min)'] = 0
                print("Warning: No sleep data columns found")
            
            # 3️⃣ สรุปข้อมูลรายวัน
            agg_dict = {}
            if 'Step' in df.columns:
                agg_dict['Step'] = 'sum'
            if 'Calorie' in df.columns:
                agg_dict['Calorie'] = 'sum'
            if 'Total Sleep (min)' in df.columns:
                agg_dict['Total Sleep (min)'] = 'sum'
            if 'Temperature' in df.columns:
                agg_dict['Temperature'] = 'mean'
            if 'Battery Level' in df.columns:
                agg_dict['Battery Level'] = 'mean'
            
            if agg_dict:
                daily_activity = df.groupby('Date').agg(agg_dict).reset_index()
                if 'Total Sleep (min)' in daily_activity.columns:
                    daily_activity['Total Sleep (hr)'] = daily_activity['Total Sleep (min)'] / 60
                
                print(f"\n=== DAILY ACTIVITY SUMMARY ===")
                print(f"Daily activity summary shape: {daily_activity.shape}")
                print(daily_activity)
                
                # แสดงสถิติพื้นฐาน
                print(f"\n=== STATISTICS ===")
                for col in daily_activity.columns:
                    if col != 'Date' and pd.api.types.is_numeric_dtype(daily_activity[col]):
                        values = daily_activity[col].dropna()
                        if len(values) > 0:
                            print(f"\n{col}:")
                            print(f"  Mean: {values.mean():.2f}")
                            print(f"  Min: {values.min():.2f}")
                            print(f"  Max: {values.max():.2f}")
                            print(f"  Std: {values.std():.2f}")
                
                # บันทึกผลลัพธ์
                output_file = f"daily_activity_{os.path.basename(folder_path)}.csv"
                daily_activity.to_csv(output_file, index=False)
                print(f"\nDaily activity data saved to: {output_file}")
                
                # แสดงข้อมูลดิบตัวอย่าง
                print(f"\n=== SAMPLE RAW DATA ===")
                sample_cols = []
                if date_col:
                    sample_cols.append(date_col)
                sample_cols.extend([col for col in ['Step', 'Calorie', 'Sleep State', 'Temperature', 'Battery Level'] if col in df.columns])
                
                if sample_cols:
                    print(df[sample_cols].head(10))
                else:
                    print(df.head(10))
                
            else:
                print("No valid columns found for aggregation!")
        else:
            print("No valid CSV files could be read!")
else:
    print("Cannot proceed: No valid folder path found!")

print("\nAnalysis completed!")

Starting sensing data analysis...
Found 119 CSV files in data_0/71_sensing_data/cefoxSR1707003701
Reading: cefoxSR17070037_activ_20210123075846.csv
  Extracted date: 2021-01-23
  Shape: (780, 11), Columns: ['Device', 'SerialNo', 'Battery Level', 'Measure Date Time', 'Temperature', 'Step', 'Calorie', 'Sleep State', 'Sleep Hour', 'Sleep Minute', 'File_Date']
Reading: cefoxSR17070037_activ_20210124000019.csv
  Extracted date: 2021-01-24
  Shape: (1440, 11), Columns: ['Device', 'SerialNo', 'Battery Level', 'Measure Date Time', 'Temperature', 'Step', 'Calorie', 'Sleep State', 'Sleep Hour', 'Sleep Minute', 'File_Date']
Reading: cefoxSR17070037_activ_20210125000019.csv
  Extracted date: 2021-01-25
  Shape: (503, 11), Columns: ['Device', 'SerialNo', 'Battery Level', 'Measure Date Time', 'Temperature', 'Step', 'Calorie', 'Sleep State', 'Sleep Hour', 'Sleep Minute', 'File_Date']
Reading: cefoxSR17070037_activ_20210126000019.csv
  Extracted date: 2021-01-26
  Shape: (388, 11), Columns: ['Device',

In [16]:
# Time Series Visualization with Seaborn-style using Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import pandas as pd
import numpy as np

# ตรวจสอบว่ามีข้อมูล daily_activity หรือไม่
if 'daily_activity' in locals() and len(daily_activity) > 0:
    print("Creating time series visualizations with Seaborn-style aesthetics using Plotly...")
    
    # Seaborn-style color palette
    seaborn_colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', 
                     '#937860', '#DA8BC3', '#8C8C8C', '#CCB974', '#64B5CD']
    
    # สร้างกราฟสำหรับแต่ละ feature
    features_to_plot = [
        ('Step', 'Daily Steps', 'Steps'),
        ('Calorie', 'Daily Calories', 'Calories'),
        ('Total Sleep (hr)', 'Daily Sleep Hours', 'Sleep Hours'),
        ('Temperature', 'Average Daily Temperature', 'Temperature (°C)'),
        ('Battery Level', 'Average Daily Battery Level', 'Battery Level (%)')
    ]
    
    for i, (feature, title, ylabel) in enumerate(features_to_plot):
        if feature in daily_activity.columns:
            fig = go.Figure()
            
            color = seaborn_colors[i % len(seaborn_colors)]
            
            # สร้างกราฟเส้นหลัก
            fig.add_trace(go.Scatter(
                x=daily_activity['Date'],
                y=daily_activity[feature],
                mode='lines+markers',
                name=feature,
                line=dict(color=color, width=3),
                marker=dict(size=8, color=color),
                hovertemplate=f'<b>Date</b>: %{{x}}<br><b>{feature}</b>: %{{y:.1f}}<extra></extra>'
            ))
            
            # เพิ่มค่าเฉลี่ย
            mean_value = daily_activity[feature].mean()
            fig.add_hline(
                y=mean_value, 
                line_dash="dash", 
                line_color='red',
                line_width=2,
                opacity=0.7,
                annotation_text=f"Average: {mean_value:.1f}",
                annotation_position="top right"
            )
            
            # ตกแต่งกราฟแบบ seaborn-style
            fig.update_layout(
                title=dict(text=title, x=0.5, font=dict(size=16, color='black', family='Arial')),
                xaxis_title="Date",
                yaxis_title=ylabel,
                template="plotly_white",
                hovermode='x unified',
                height=500,
                font=dict(family="Arial", size=12),
                plot_bgcolor='white',
                paper_bgcolor='white',
                xaxis=dict(
                    showgrid=True, 
                    gridwidth=1, 
                    gridcolor='lightgray',
                    showline=True,
                    linewidth=1,
                    linecolor='black'
                ),
                yaxis=dict(
                    showgrid=True, 
                    gridwidth=1, 
                    gridcolor='lightgray',
                    showline=True,
                    linewidth=1,
                    linecolor='black'
                )
            )
            
            fig.show()
            print(f"{feature}: Range from {daily_activity[feature].min():.1f} to {daily_activity[feature].max():.1f}")
        else:
            print(f"Feature '{feature}' not available in data")
    
    # สร้างกราฟรวมทุก features ในหน้าเดียวแบบ seaborn FacetGrid style
    available_features = [f for f, _, _ in features_to_plot if f in daily_activity.columns]
    
    if len(available_features) > 1:
        print("\\nCreating FacetGrid-style combined visualization...")
        
        # สร้าง subplots
        rows = (len(available_features) + 1) // 2  # 2 columns
        cols = 2
        
        fig = make_subplots(
            rows=rows, 
            cols=cols,
            subplot_titles=[f"{f} Over Time" for f, _, _ in features_to_plot if f in daily_activity.columns],
            vertical_spacing=0.12,
            horizontal_spacing=0.1
        )
        
        for i, (feature, title, ylabel) in enumerate(features_to_plot):
            if feature in daily_activity.columns:
                row = (i // 2) + 1
                col = (i % 2) + 1
                
                color = seaborn_colors[i % len(seaborn_colors)]
                
                # เพิ่มกราฟหลัก
                fig.add_trace(
                    go.Scatter(
                        x=daily_activity['Date'],
                        y=daily_activity[feature],
                        mode='lines+markers',
                        name=feature,
                        line=dict(color=color, width=2),
                        marker=dict(size=6, color=color),
                        hovertemplate=f'<b>{feature}</b>: %{{y:.1f}}<extra></extra>',
                        showlegend=False
                    ),
                    row=row, col=col
                )
                
                # เพิ่มเส้นค่าเฉลี่ย
                mean_value = daily_activity[feature].mean()
                fig.add_hline(
                    y=mean_value,
                    line_dash="dash",
                    line_color='red',
                    opacity=0.7,
                    row=row, col=col
                )
                
                # ตั้งชื่อแกน Y
                fig.update_yaxes(title_text=ylabel, row=row, col=col)
        
        # ตกแต่งกราฟรวมแบบ seaborn style
        fig.update_layout(
            title=dict(text="📊 Daily Activity Time Series - Seaborn Style", x=0.5, font=dict(size=18)),
            template="plotly_white",
            height=400 * rows,
            font=dict(family="Arial", size=11),
            hovermode='x unified'
        )
        
        fig.update_xaxes(title_text="Date")
        fig.show()
        
        # สร้างกราฟ correlation heatmap ด้วย plotly
        if len(available_features) > 2:
            print("\\nCreating correlation heatmap...")
            
            correlation_data = daily_activity[available_features].corr()
            
            fig = go.Figure(data=go.Heatmap(
                z=correlation_data.values,
                x=correlation_data.columns,
                y=correlation_data.columns,
                colorscale='RdBu',
                zmid=0,
                text=correlation_data.round(2).values,
                texttemplate="%{text}",
                textfont={"size": 12},
                hovertemplate='<b>%{x} vs %{y}</b><br>Correlation: %{z:.3f}<extra></extra>'
            ))
            
            fig.update_layout(
                title='Feature Correlation Heatmap - Seaborn Style',
                xaxis_title="Features",
                yaxis_title="Features",
                template="plotly_white",
                height=500,
                width=500,
                font=dict(family="Arial", size=12)
            )
            
            fig.show()
    
    print("\\nSeaborn-style visualization completed using Plotly!")
else:
    print("No daily activity data available for visualization!")

Creating time series visualizations with Seaborn-style aesthetics using Plotly...


Step: Range from 0.0 to 3014.0


Calorie: Range from 928.0 to 7703.0


Total Sleep (hr): Range from 0.0 to 1786.8


Temperature: Range from 2.4 to 8.4


Battery Level: Range from 56.0 to 81.5
\nCreating FacetGrid-style combined visualization...


\nCreating correlation heatmap...


\nSeaborn-style visualization completed using Plotly!


In [17]:
# Alternative: Simple Text-based Time Series Analysis
if 'daily_activity' in locals() and len(daily_activity) > 0:
    print("\n" + "="*60)
    print("TEXT-BASED TIME SERIES ANALYSIS")
    print("="*60)
    
    features = ['Step', 'Calorie', 'Total Sleep (hr)', 'Temperature', 'Battery Level']
    
    for feature in features:
        if feature in daily_activity.columns:
            print(f"\n📊 {feature.upper()} OVER TIME:")
            print("-" * 40)
            
            # แสดงข้อมูลแต่ละวัน
            for _, row in daily_activity.iterrows():
                date = row['Date']
                value = row[feature]
                
                # สร้างแถบกราฟด้วยอักขระ
                if pd.notna(value):
                    # ปรับสเกลให้อยู่ในช่วง 0-50 อักขระ
                    if feature in ['Step', 'Calorie']:
                        # สำหรับค่าที่สูง ใช้สเกลต่างกัน
                        max_val = daily_activity[feature].max()
                        bar_length = int((value / max_val) * 50) if max_val > 0 else 0
                    else:
                        # สำหรับค่าที่ต่ำกว่า
                        if feature == 'Total Sleep (hr)':
                            bar_length = int((value / 12) * 50)  # สมมติการนอนสูงสุด 12 ชั่วโมง
                        elif feature == 'Temperature':
                            bar_length = int(((value - 30) / 10) * 50) if value >= 30 else 0  # อุณหภูมิ 30-40°C
                        else:  # Battery Level
                            bar_length = int((value / 100) * 50)  # 0-100%
                    
                    bar_length = max(0, min(50, bar_length))  # จำกัดให้อยู่ในช่วง 0-50
                    bar = "█" * bar_length + "░" * (50 - bar_length)
                    print(f"{date}: {bar} {value:.1f}")
                else:
                    print(f"{date}: {'░' * 50} N/A")
            
            # แสดงสถิติ
            values = daily_activity[feature].dropna()
            if len(values) > 0:
                print(f"\nStatistics:")
                print(f"  Average: {values.mean():.2f}")
                print(f"  Trend: ", end="")
                
                # คำนวณ trend แบบง่าย
                if len(values) >= 2:
                    first_half = values[:len(values)//2].mean()
                    second_half = values[len(values)//2:].mean()
                    if second_half > first_half * 1.05:
                        print("📈 Increasing")
                    elif second_half < first_half * 0.95:
                        print("📉 Decreasing")
                    else:
                        print("➡️ Stable")
                else:
                    print("➡️ Insufficient data")
    
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    
    # สรุปข้อมูลที่สำคัญ
    for feature in features:
        if feature in daily_activity.columns:
            values = daily_activity[feature].dropna()
            if len(values) > 0:
                print(f"{feature}: Avg={values.mean():.1f}, Min={values.min():.1f}, Max={values.max():.1f}")

else:
    print("No daily activity data available for text analysis!")


TEXT-BASED TIME SERIES ANALYSIS

📊 STEP OVER TIME:
----------------------------------------
2021-01-21: ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 0.0
2021-01-22: ████████████████████████░░░░░░░░░░░░░░░░░░░░░░░░░░ 1498.0
2021-01-23: ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 402.0
2021-01-24: ████████████████████████████████████████░░░░░░░░░░ 2451.0
2021-01-25: █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 86.0
2021-01-26: ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 396.0
2021-01-27: ███████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 687.0
2021-01-28: █████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 590.0
2021-01-29: █████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 584.0
2021-01-30: ██████████████████████████████████████████████████ 3014.0
2021-01-31: █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 72.0

Statistics:
  Average: 889.09
  Trend: ➡️ Stable

📊 CALORIE OVER TIME:
----------------------------------------
2021-01-21: ███████████░░░░░░░░░░░░░

In [20]:
# Advanced Seaborn-style Visualizations using Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.figure_factory as ff
import numpy as np

try:
    if 'daily_activity' in locals() and len(daily_activity) > 0:
        print("Creating advanced seaborn-style visualizations with Plotly...")
        
        # Seaborn color palettes
        seaborn_colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
        seaborn_qualitative = px.colors.qualitative.Set2
        
        # เตรียมข้อมูล
        features_config = [
            ('Step', 'Daily Steps', 'Steps'),
            ('Calorie', 'Daily Calories', 'Calories'),
            ('Total Sleep (hr)', 'Daily Sleep Hours', 'Hours'),
            ('Temperature', 'Average Daily Temperature', 'Temperature (°C)'),
            ('Battery Level', 'Average Daily Battery Level', 'Battery (%)')
        ]
        
        available_features = [(f, t, y) for f, t, y in features_config if f in daily_activity.columns]
        
        if available_features:
            # 1. Distribution plots (violin + box plots) สำหรับแต่ละ feature
            print("Creating distribution plots (violin + box style)...")
            
            feature_names = [f for f, _, _ in available_features]
            
            # เตรียมข้อมูลแบบ melted
            melted_data = daily_activity[feature_names].melt(var_name='Feature', value_name='Value')
            
            # Violin plot + Box plot combination
            fig = go.Figure()
            
            for i, feature in enumerate(feature_names):
                feature_data = daily_activity[feature].dropna()
                color = seaborn_colors[i % len(seaborn_colors)]
                
                # Violin plot
                fig.add_trace(go.Violin(
                    y=feature_data,
                    name=feature,
                    box_visible=True,
                    meanline_visible=True,
                    fillcolor=color,
                    opacity=0.6,
                    line_color='black',
                    hoveron="points",
                    hovertemplate=f'<b>{feature}</b><br>Value: %{{y:.1f}}<extra></extra>'
                ))
            
            fig.update_layout(
                title='Feature Distributions (Violin + Box Plots) - Seaborn Style',
                yaxis_title="Values",
                template="plotly_white",
                height=600,
                font=dict(family="Arial", size=12),
                showlegend=True
            )
            
            fig.show()
            
            # 2. Time series with confidence intervals and trend lines
            print("Creating time series with trend analysis...")
            
            # เพิ่มคอลัมน์ day number สำหรับ trend analysis
            daily_activity_copy = daily_activity.copy()
            daily_activity_copy['Day_Number'] = range(len(daily_activity_copy))
            
            for i, (feature, title, ylabel) in enumerate(available_features):
                fig = go.Figure()
                color = seaborn_colors[i % len(seaborn_colors)]
                
                # กราฟเส้นหลัก
                fig.add_trace(go.Scatter(
                    x=daily_activity_copy['Day_Number'],
                    y=daily_activity_copy[feature],
                    mode='lines+markers',
                    name=feature,
                    line=dict(color=color, width=3),
                    marker=dict(size=8, color=color),
                    hovertemplate=f'<b>Day</b>: %{{x}}<br><b>{feature}</b>: %{{y:.1f}}<extra></extra>'
                ))
                
                # เพิ่ม trend line ด้วย numpy polyfit
                x_vals = daily_activity_copy['Day_Number'].values
                y_vals = daily_activity_copy[feature].values
                
                # คำนวณ trend line
                z = np.polyfit(x_vals, y_vals, 1)
                p = np.poly1d(z)
                trend_y = p(x_vals)
                
                fig.add_trace(go.Scatter(
                    x=x_vals,
                    y=trend_y,
                    mode='lines',
                    name=f'{feature} Trend',
                    line=dict(color='red', width=2, dash='dash'),
                    opacity=0.8,
                    hovertemplate=f'<b>Trend</b>: %{{y:.1f}}<extra></extra>'
                ))
                
                # คำนวณ confidence interval (mock)
                std_dev = np.std(y_vals)
                upper_bound = trend_y + std_dev
                lower_bound = trend_y - std_dev
                
                fig.add_trace(go.Scatter(
                    x=x_vals,
                    y=upper_bound,
                    mode='lines',
                    line=dict(width=0),
                    showlegend=False,
                    hoverinfo='skip'
                ))
                
                fig.add_trace(go.Scatter(
                    x=x_vals,
                    y=lower_bound,
                    mode='lines',
                    line=dict(width=0),
                    fill='tonexty',
                    fillcolor=f'rgba({int(color[1:3], 16)}, {int(color[3:5], 16)}, {int(color[5:7], 16)}, 0.2)',
                    name='Confidence Interval',
                    showlegend=False,
                    hoverinfo='skip'
                ))
                
                # ตกแต่งกราห
                fig.update_layout(
                    title=f'{title} with Trend Analysis - Seaborn Style',
                    xaxis_title='Day Number',
                    yaxis_title=ylabel,
                    template="plotly_white",
                    height=500,
                    font=dict(family="Arial", size=12)
                )
                
                # ปรับ x-axis labels เป็นวันที่จริง
                day_labels = [str(date)[-5:] for date in daily_activity['Date']]
                fig.update_xaxes(
                    tickvals=list(range(0, len(day_labels), max(1, len(day_labels)//5))),
                    ticktext=[day_labels[i] for i in range(0, len(day_labels), max(1, len(day_labels)//5))]
                )
                
                fig.show()
            
            # 3. Pairplot-style scatter matrix
            if len(available_features) >= 3:
                print("Creating pairplot-style scatter matrix...")
                
                feature_names = [f for f, _, _ in available_features]
                
                # สร้าง scatter matrix
                fig = ff.create_scatterplotmatrix(
                    daily_activity[feature_names], 
                    diag='histogram',
                    height=800, 
                    width=800,
                    colormap='Viridis',
                    colormap_type='cat'
                )
                
                fig.update_layout(
                    title='Feature Relationships Scatter Matrix - Seaborn Style',
                    font=dict(family="Arial", size=10)
                )
                
                fig.show()
            
            # 4. Advanced heatmap with clustering effect
            if len(available_features) > 2:
                print("Creating advanced correlation heatmap...")
                
                feature_names = [f for f, _, _ in available_features]
                correlation_matrix = daily_activity[feature_names].corr()
                
                # สร้างกราฟ heatmap ที่สวยงาม
                fig = go.Figure(data=go.Heatmap(
                    z=correlation_matrix.values,
                    x=correlation_matrix.columns,
                    y=correlation_matrix.columns,
                    colorscale='RdBu',
                    zmid=0,
                    text=correlation_matrix.round(3).values,
                    texttemplate="%{text}",
                    textfont={"size": 14, "color": "white"},
                    hovertemplate='<b>%{x} vs %{y}</b><br>Correlation: %{z:.3f}<extra></extra>',
                    colorbar=dict(title="Correlation Coefficient", titleside="right")
                ))
                
                fig.update_layout(
                    title='Feature Correlation Matrix - Seaborn Style',
                    xaxis_title="Features",
                    yaxis_title="Features",
                    template="plotly_white",
                    height=600,
                    width=600,
                    font=dict(family="Arial", size=12),
                    xaxis=dict(side="bottom"),
                    yaxis=dict(side="left")
                )
                
                fig.show()
            
            # 5. Multi-line interactive plot ด้วย seaborn aesthetics
            print("Creating interactive multi-feature comparison...")
            
            fig = go.Figure()
            
            for i, (feature, title, ylabel) in enumerate(available_features):
                color = seaborn_colors[i % len(seaborn_colors)]
                
                fig.add_trace(go.Scatter(
                    x=daily_activity['Date'],
                    y=daily_activity[feature],
                    mode='lines+markers',
                    name=feature,
                    line=dict(color=color, width=3),
                    marker=dict(size=8, color=color),
                    hovertemplate=f'<b>Date</b>: %{{x}}<br><b>{feature}</b>: %{{y:.1f}}<extra></extra>'
                ))
            
            fig.update_layout(
                title=dict(text="📊 Interactive Multi-Feature Comparison - Seaborn Style", 
                          x=0.5, font=dict(size=18)),
                xaxis_title="Date",
                yaxis_title="Values (Normalized Scale)",
                template="plotly_white",
                hovermode='x unified',
                height=600,
                font=dict(family="Arial", size=12),
                legend=dict(
                    orientation="h",
                    yanchor="bottom",
                    y=1.02,
                    xanchor="right",
                    x=1
                )
            )
            
            fig.show()
            
            # 6. Statistical summary table
            print("\\nCreating statistical summary...")
            
            summary_stats = daily_activity[feature_names].describe().round(2)
            
            fig = go.Figure(data=[go.Table(
                header=dict(values=['Statistic'] + list(feature_names),
                           fill_color='lightblue',
                           align='left',
                           font=dict(size=12, color='black')),
                cells=dict(values=[summary_stats.index] + [summary_stats[col] for col in feature_names],
                          fill_color='white',
                          align='left',
                          font=dict(size=11))
            )])
            
            fig.update_layout(
                title='Statistical Summary Table - Seaborn Style',
                height=400,
                font=dict(family="Arial", size=12)
            )
            
            fig.show()
            
            print("\\n🎉 All advanced seaborn-style visualizations created successfully!")
            
        else:
            print("❌ No features available for plotting!")
    else:
        print("❌ No daily_activity data available for plotting!")

except Exception as e:
    print(f"❌ Error creating visualizations: {e}")
    import traceback
    traceback.print_exc()

Creating advanced seaborn-style visualizations with Plotly...
Creating distribution plots (violin + box style)...


Creating time series with trend analysis...


Creating pairplot-style scatter matrix...


Creating advanced correlation heatmap...


Creating interactive multi-feature comparison...


\nCreating statistical summary...


\n🎉 All advanced seaborn-style visualizations created successfully!


In [19]:
# Detailed Time Series with Seaborn-style Aesthetics (Hourly/Minute Level)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import numpy as np
import pandas as pd

try:
    if 'df' in locals() and len(df) > 0:
        print("Creating detailed seaborn-style time series graphs (raw data)...")
        
        # Seaborn color palette
        seaborn_colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', 
                         '#937860', '#DA8BC3', '#8C8C8C', '#CCB974', '#64B5CD']
        
        # ใช้ข้อมูลดิบแทนการรวมรายวัน
        raw_data = df.copy()
        
        # ตรวจสอบ datetime column
        datetime_col = None
        for col in raw_data.columns:
            if 'date' in col.lower() and 'time' in col.lower():
                datetime_col = col
                break
        
        if datetime_col and datetime_col in raw_data.columns:
            # ใช้ datetime แบบเต็ม (วันที่ + เวลา)
            raw_data[datetime_col] = pd.to_datetime(raw_data[datetime_col], errors='coerce')
            raw_data = raw_data.dropna(subset=[datetime_col])
            raw_data = raw_data.sort_values(datetime_col)
            
            print(f"Data time range: {raw_data[datetime_col].min()} to {raw_data[datetime_col].max()}")
            print(f"Total data points: {len(raw_data)}")
            
            # เตรียมข้อมูลสำหรับแต่ละ feature
            features_config = [
                ('Step', 'Step Count Over Time', 'Steps'),
                ('Calorie', 'Calorie Burn Over Time', 'Calories'),
                ('Sleep Hour', 'Sleep Hours Over Time', 'Hours'),
                ('Temperature', 'Temperature Over Time', 'Temperature (°C)'),
                ('Battery Level', 'Battery Level Over Time', 'Battery (%)')
            ]
            
            # กรองเฉพาะ features ที่มีข้อมูล
            available_features = []
            for feature, title, ylabel in features_config:
                if feature in raw_data.columns:
                    # ทำความสะอาดข้อมูล
                    raw_data[feature] = pd.to_numeric(raw_data[feature], errors='coerce')
                    non_null_count = raw_data[feature].notna().sum()
                    if non_null_count > 0:
                        available_features.append((feature, title, ylabel))
                        print(f"✅ {feature}: {non_null_count} valid data points")
                    else:
                        print(f"❌ {feature}: No valid data")
            
            if available_features:
                # 1. กราฟแยกแต่ละ feature ด้วยสไตล์ seaborn
                print("Creating individual feature plots with Seaborn aesthetics...")
                
                for i, (feature, title, ylabel) in enumerate(available_features):
                    # ข้อมูลที่ไม่เป็น null
                    mask = raw_data[feature].notna()
                    x_data = raw_data.loc[mask, datetime_col]
                    y_data = raw_data.loc[mask, feature]
                    
                    if len(y_data) > 0:
                        fig = go.Figure()
                        color = seaborn_colors[i % len(seaborn_colors)]
                        
                        # สร้างกราฟเส้นหลัก
                        fig.add_trace(go.Scatter(
                            x=x_data,
                            y=y_data,
                            mode='lines',
                            name=feature,
                            line=dict(color=color, width=2),
                            opacity=0.8,
                            hovertemplate=f'<b>Time</b>: %{{x}}<br><b>{feature}</b>: %{{y:.1f}}<extra></extra>'
                        ))
                        
                        # เพิ่มเส้นค่าเฉลี่ย
                        mean_value = y_data.mean()
                        fig.add_hline(
                            y=mean_value,
                            line_dash="dash",
                            line_color='red',
                            line_width=2,
                            opacity=0.7,
                            annotation_text=f'Average: {mean_value:.1f}',
                            annotation_position="top right"
                        )
                        
                        # เพิ่ม smoothed trend line
                        if len(y_data) > 10:
                            # Rolling average for smoother trend
                            window_size = max(5, len(y_data) // 20)
                            y_smooth = y_data.rolling(window=window_size, center=True).mean()
                            
                            fig.add_trace(go.Scatter(
                                x=x_data,
                                y=y_smooth,
                                mode='lines',
                                name=f'{feature} Trend',
                                line=dict(color='orange', width=3, dash='dot'),
                                opacity=0.8,
                                hovertemplate=f'<b>Trend</b>: %{{y:.1f}}<extra></extra>'
                            ))
                        
                        # ตกแต่งกราฟแบบ seaborn
                        fig.update_layout(
                            title=dict(text=title, x=0.5, font=dict(size=16, family='Arial')),
                            xaxis_title="Date & Time",
                            yaxis_title=ylabel,
                            template="plotly_white",
                            hovermode='x unified',
                            height=500,
                            font=dict(family="Arial", size=12),
                            plot_bgcolor='white',
                            paper_bgcolor='white',
                            xaxis=dict(
                                showgrid=True, 
                                gridwidth=1, 
                                gridcolor='lightgray',
                                showline=True,
                                linewidth=1,
                                linecolor='black'
                            ),
                            yaxis=dict(
                                showgrid=True, 
                                gridwidth=1, 
                                gridcolor='lightgray',
                                showline=True,
                                linewidth=1,
                                linecolor='black'
                            )
                        )
                        
                        fig.show()
                        print(f"📊 {feature}: {len(y_data)} points, Range: {y_data.min():.1f} - {y_data.max():.1f}")
                
                # 2. กราฟรวมแบบ subplot ด้วย seaborn aesthetics
                if len(available_features) > 1:
                    print(f"\\nCreating combined subplot with {len(available_features)} features using Seaborn style...")
                    
                    fig = make_subplots(
                        rows=len(available_features),
                        cols=1,
                        subplot_titles=[title for _, title, _ in available_features],
                        vertical_spacing=0.08,
                        shared_xaxes=True
                    )
                    
                    for i, (feature, title, ylabel) in enumerate(available_features):
                        mask = raw_data[feature].notna()
                        x_data = raw_data.loc[mask, datetime_col]
                        y_data = raw_data.loc[mask, feature]
                        
                        if len(y_data) > 0:
                            color = seaborn_colors[i % len(seaborn_colors)]
                            
                            # กราฟหลัก
                            fig.add_trace(
                                go.Scatter(
                                    x=x_data,
                                    y=y_data,
                                    mode='lines',
                                    name=feature,
                                    line=dict(color=color, width=1.5),
                                    hovertemplate=f'<b>{feature}</b>: %{{y:.1f}}<extra></extra>',
                                    showlegend=False
                                ),
                                row=i+1, col=1
                            )
                            
                            # เส้นค่าเฉลี่ย
                            mean_value = y_data.mean()
                            fig.add_hline(
                                y=mean_value,
                                line_dash="dash",
                                line_color='red',
                                opacity=0.6,
                                row=i+1, col=1
                            )
                            
                            fig.update_yaxes(title_text=ylabel, row=i+1, col=1)
                    
                    # ตกแต่งกราฟรวมแบบ seaborn
                    fig.update_layout(
                        title=dict(text="🔍 Detailed Time Series - Seaborn Style (Raw Data)", 
                                  x=0.5, font=dict(size=18, family='Arial')),
                        template="plotly_white",
                        height=300 * len(available_features),
                        font=dict(family="Arial", size=12),
                        hovermode='x unified'
                    )
                    
                    fig.update_xaxes(title_text="Date & Time", row=len(available_features), col=1)
                    fig.show()
                
                # 3. กราฟ Activity Pattern heatmap (แบ่งตามชั่วโมงในวัน)
                if any(f in raw_data.columns for f in ['Step', 'Calorie']):
                    print("\\nCreating activity pattern heatmap with Seaborn aesthetics...")
                    
                    # เพิ่มคอลัมน์ชั่วโมงและวัน
                    raw_data['Hour'] = raw_data[datetime_col].dt.hour
                    raw_data['DayOfWeek'] = raw_data[datetime_col].dt.day_name()
                    raw_data['Date_Only'] = raw_data[datetime_col].dt.date
                    
                    # สำหรับ Steps
                    if 'Step' in raw_data.columns:
                        # สร้าง pivot table สำหรับ heatmap
                        heatmap_data = raw_data.pivot_table(
                            values='Step', 
                            index='Date_Only', 
                            columns='Hour', 
                            aggfunc='mean'
                        )
                        
                        fig = go.Figure(data=go.Heatmap(
                            z=heatmap_data.values,
                            x=[f"{h}:00" for h in heatmap_data.columns],
                            y=[str(d) for d in heatmap_data.index],
                            colorscale='Viridis',
                            hoverongaps=False,
                            hovertemplate='<b>Date</b>: %{y}<br><b>Hour</b>: %{x}<br><b>Avg Steps</b>: %{z:.1f}<extra></extra>'
                        ))
                        
                        fig.update_layout(
                            title='🔥 Activity Heatmap: Steps by Date and Hour - Seaborn Style',
                            xaxis_title='Hour of Day',
                            yaxis_title='Date',
                            template="plotly_white",
                            height=600,
                            font=dict(family="Arial", size=12)
                        )
                        
                        fig.show()
                
                # 4. Hourly pattern analysis with error bars
                activity_features = ['Step', 'Calorie']
                available_activity = [f for f in activity_features if f in raw_data.columns]
                
                if available_activity:
                    print("\\nCreating hourly pattern analysis with confidence intervals...")
                    
                    for feature in available_activity:
                        hourly_stats = raw_data.groupby('Hour')[feature].agg(['mean', 'std', 'count']).reset_index()
                        hourly_stats['se'] = hourly_stats['std'] / np.sqrt(hourly_stats['count'])  # Standard error
                        
                        fig = go.Figure()
                        
                        # Bar plot สำหรับค่าเฉลี่ย
                        fig.add_trace(go.Bar(
                            x=hourly_stats['Hour'],
                            y=hourly_stats['mean'],
                            name=f'Average {feature}',
                            marker_color='lightblue',
                            opacity=0.7,
                            hovertemplate=f'<b>Hour</b>: %{{x}}:00<br><b>Avg {feature}</b>: %{{y:.1f}}<extra></extra>'
                        ))
                        
                        # Error bars
                        fig.add_trace(go.Scatter(
                            x=hourly_stats['Hour'],
                            y=hourly_stats['mean'] + hourly_stats['se'],
                            mode='markers',
                            marker=dict(size=0.1, color='rgba(0,0,0,0)'),
                            showlegend=False,
                            hoverinfo='skip'
                        ))
                        
                        fig.add_trace(go.Scatter(
                            x=hourly_stats['Hour'],
                            y=hourly_stats['mean'] - hourly_stats['se'],
                            mode='markers',
                            marker=dict(size=0.1, color='rgba(0,0,0,0)'),
                            fill='tonexty',
                            fillcolor='rgba(255,0,0,0.2)',
                            name='Confidence Interval',
                            hoverinfo='skip'
                        ))
                        
                        fig.update_layout(
                            title=f'🚶‍♂️ {feature} Pattern by Hour with Confidence Intervals - Seaborn Style',
                            xaxis_title='Hour of Day (0-23)',
                            yaxis_title=f'Average {feature}',
                            template="plotly_white",
                            height=500,
                            font=dict(family="Arial", size=12)
                        )
                        
                        fig.show()
                
                print("\\n🎉 All detailed seaborn-style visualizations created successfully!")
                print(f"📊 Data resolution: Individual measurements (minute-level)")
                print(f"⏰ Time range: {raw_data[datetime_col].min()} to {raw_data[datetime_col].max()}")
                
            else:
                print("❌ No valid features found for detailed plotting!")
        
        else:
            print("❌ No datetime column found! Cannot create detailed time series.")
            print("Available columns:", raw_data.columns.tolist())
    
    else:
        print("❌ No raw data available for detailed plotting!")

except Exception as e:
    print(f"❌ Error creating detailed seaborn-style graphs: {e}")
    import traceback
    traceback.print_exc()

Creating detailed seaborn-style time series graphs (raw data)...
Data time range: 2021-01-21 14:49:36 to 2021-01-31 09:16:00
Total data points: 5306
✅ Step: 5306 valid data points
✅ Calorie: 5306 valid data points
✅ Sleep Hour: 5306 valid data points
✅ Temperature: 5306 valid data points
✅ Battery Level: 5306 valid data points
Creating individual feature plots with Seaborn aesthetics...


C:\Users\punch\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\_plotly_utils\basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



📊 Step: 5306 points, Range: 0.0 - 127.0


📊 Calorie: 5306 points, Range: 5.0 - 34.0


📊 Sleep Hour: 5306 points, Range: 0.0 - 5.0


📊 Temperature: 5306 points, Range: -4.0 - 14.0


📊 Battery Level: 5306 points, Range: 0.0 - 100.0
\nCreating combined subplot with 5 features using Seaborn style...


\nCreating activity pattern heatmap with Seaborn aesthetics...


\nCreating hourly pattern analysis with confidence intervals...


\n🎉 All detailed seaborn-style visualizations created successfully!
📊 Data resolution: Individual measurements (minute-level)
⏰ Time range: 2021-01-21 14:49:36 to 2021-01-31 09:16:00
